In [ ]:
!pip -q install amplpy sympy numpy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 30.4 MB/s eta 0:00:00


In [ ]:
# 1. Instalación e inicialización de AMPL con tu licencia Community Edition
from amplpy import ampl_notebook

# Inicializa AMPL en Colab con tu UUID (Community Edition) Inicializar AMPL
ampl = ampl_notebook(
    modules=["highs", "cbc", "gurobi", "cplex", "coin"],  # solvers disponibles
    license_uuid="936b618d-a013-406f-9809-49679f557c26"
)

Licensed to AMPL Academic Community Edition License for <m.godoyseplveda@uandresbello.edu>.


In [ ]:
%%writefile 18_4_4.mod
set M := 1..3;                     # meses

param dem {M};                     # requerimiento
param K   {M};                     # costo de preparacion
param c   {M};                     # costo unitario en horas normales
param h   := 2;                    # costo de inventario por unidad
param U_n := 3;                    # maximo normal
param U_o := 1;                    # maximo horas extra
param alpha := 2;                  # factor costo extra (= 2 * c_m)

param I0  := 1;                    # inventario inicial
param If  := 2;                    # inventario deseado fin mes 3

# ------------------- variables -------------------------
var x {M} integer >= 0 <= U_n;      # produccion normal
var y {M} binary;                   # 1 si se produce normal
var o {M} integer >= 0 <= U_o;      # produccion horas extra
var inv{M} integer >= 0;            # inventario final mes m

# ------------------- restricciones ---------------------
s.t. SetupLimit {m in M}: x[m] <= U_n * y[m];

s.t. Balance {m in M}:
      (if m=1 then I0 else inv[m-1])
    + x[m] + o[m] - dem[m] = inv[m];

s.t. FinalInv: inv[3] = If;

# ------------------- funcion objetivo ------------------
minimize TotalCost:
    sum{m in M} ( K[m]*y[m] + c[m]*x[m] + alpha*c[m]*o[m] + h*inv[m] );


Overwriting 18_4_4.mod


In [ ]:
%%writefile 18_4_4.dat
param dem :=
 1 1
 2 3
 3 2 ;

param K :=
 1 5
 2 10
 3 5 ;

param c :=
 1 8
 2 10
 3 9 ;


Overwriting 18_4_4.dat


In [ ]:
ampl.read('18_4_4.mod')
ampl.readData('18_4_4.dat')

ampl.option['solver'] = 'highs'     # CBC produce la misma solución
ampl.solve()

ampl.display('x','o','inv','y','TotalCost')


HiGHS 1.11.0: HiGHS 1.11.0: optimal solution; objective 89
2 simplex iterations
1 branching nodes
:   x   o inv   y    :=
1   3   0   3   1
2   0   0   0   0
3   3   1   2   1
;

TotalCost = 89

